# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup
First, let's print a simple message to ensure our environment is set up correctly.

In [5]:
print("Hello World")

Hello World
Name: black
Version: 24.4.2
Summary: The uncompromising code formatter.
Home-page: 
Author: 
Author-email: Łukasz Langa <lukasz@langa.pl>
License: MIT
Location: /nfs/students/daro/miniconda3/envs/env-quant-rel/lib/python3.12/site-packages
Requires: click, mypy-extensions, packaging, pathspec, platformdirs
Required-by: 


## 2. Checking System Memory
We will check the available system memory to ensure that we have enough resources to load and run the model. The following command outputs the total, free, and available memory in gigabytes.

In [2]:
# !pip show transformers
# !pip show torch
# !pip show python-dotenv
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'

Name: python-dotenv
Version: 1.0.1
Summary: Read key-value pairs from a .env file and set them as environment variables
Home-page: https://github.com/theskumar/python-dotenv
Author: Saurabh Kumar
Author-email: me+github@saurabh-kumar.com
License: BSD-3-Clause
Location: /nfs/students/daro/miniconda3/envs/env-quant-rel/lib/python3.12/site-packages
Requires: 
Required-by: 
MemTotal: 251.77 GB
MemFree: 197.58 GB
MemAvailable: 232.45 GB


## 3. Code Formatting and Linting
We use `black` for code formatting and `pylint` for linting to ensure our code is clean and follows best practices.


In [6]:
!black notebooks/Llama-3-8B-quant.ipynb
!pylint notebooks/Llama-3-8B-quant.ipynb

All done! ✨ 🍰 ✨
1 file left unchanged.
************* Module Llama-3-8B-quant
notebooks/Llama-3-8B-quant.ipynb:9:0: C0301: Line too long (182/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:39:0: C0301: Line too long (196/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:61:0: C0301: Line too long (239/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:70:0: C0301: Line too long (123/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:95:0: C0301: Line too long (136/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:96:0: C0301: Line too long (101/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:123:0: C0301: Line too long (167/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:137:0: C0301: Line too long (237/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:170:0: C0301: Line too long (148/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:204:0: C0301: Line too long (159/100) (line-too-long)
notebooks/Llama-3-8B-quant.ipynb:235:0: C0301: Line too long

## 4. Loading Environment Variables
We load the Hugging Face token from an environment variable to authenticate our session. This token is necessary to access the model from the Hugging Face Hub.


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the .env file
load_dotenv()

# Read the Hugging Face token from the environment variable
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

# Log in using the token
login(token=huggingface_token)

Hugging Face token loaded successfully.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /nfs/homedirs/daro/.cache/huggingface/token
Login successful


## 5. Checking CUDA Availability
We check if CUDA is available on the system. CUDA is essential for running the model on GPU, which significantly speeds up the computations.


In [3]:
import torch

# Check CUDA availability
if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA GeForce GTX 1080 Ti


### 6.1 TinyLlama-1.1B

In [9]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])
# <|system|>
# You are a friendly chatbot who always responds in the style of a pirate.</s>
# <|user|>
# How many helicopters can a human eat in one sitting?</s>
# <|assistant|>
# ...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/nfs/students/daro/miniconda3/envs/env-quant-rel/lib/python3.12/site-packages/accelerate/utils/modeling.py:1393: UserWarning: Current model requires 1408 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>
<|assistant|>
I don't have information about the specific human population, but a human can eat anywhere from 10 to 15 servings of food in one sitting. However, eating too much can lead to stomach cramps, nausea, and diarrhea, so it's best to limit your intake to 10-15 servings per sitting.


## 6.3. AutoModelForCausalLM Generation for Llama-3-8B

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device="cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)

In [26]:
print(model.get_memory_footprint())

4400196352


In [3]:
# input_text = "Once upon a time, a curious fox..."
input_text = "What famous tower is in Paris?"

# Encode input text to tensor and move to device
input_ids = tokenizer(input_text, return_tensors="pt").to(device)

generated_ids = model.generate(
    input_ids=input_ids["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

# Decode generated IDs back to text
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Print generated text
print("Generated Text:", generated_text)

Generated Text: What famous tower is in Paris?

Student: The Eiffel Tower.

Teacher: The Eiffel Tower is a famous tower located in Paris, France. It was designed by Gustave Eiffel for the 1889 World's Fair and was completed in 1889. The tower stands at 324 meters (1,063 feet) tall, making it the tallest structure in Paris and one of the tallest in the world.

Student: Wow, the Eiffel Tower is so tall!

Teacher: Yes, it's truly a marvel of engineering and architecture. The tower is made of wrought iron and steel, and it's designed to withstand earthquakes and other natural disasters. It's a popular tourist attraction, and millions of people visit the tower each year.

Student: Wow, I've heard of the Eiffel Tower before, but I didn't know it was so tall.

Teacher: Yes, the Eiffel Tower is one of the most recognizable landmarks in the world. It's a symbol of France and a testament to the ingenuity and creativity of Gustave Eiffel and his team.

Student: It's amazing how the Eiffel Tower h

In [4]:
import json

results = {"input": input_text, "output": generated_text}
with open("results.json", "w") as f:
    json.dump(results, f)

## 7. Quantization

In [2]:
# Setting quantization bits for the model
weight_quantization_bits = 8
double_quant = False

In [25]:
#!pip index versions accelerate
#!pip install accelerate --force-reinstall
!pip install --upgrade transformers accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
import torch
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig, AwqConfig
from accelerate.utils import load_and_quantize_model

bnb_config = BitsAndBytesConfig(
    load_in_8bit=(weight_quantization_bits == 8),
    load_in_4bit=(weight_quantization_bits == 4),
    llm_int8_threshold=6.0,
    llm_int8_skip_modules=["lm_head"],
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
    # bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    # bnb_4bit_quant_type="nf4"
    bnb_4bit_use_double_quant=double_quant,
    skip_modules=[],
)

awq_config = AwqConfig()

smashed_model_bnb = load_and_quantize_model(model, bnb_quantization_config=bnb_config, device_map=device)
print(smashed_model_bnb.__class__.__name__)

# Calibration Dataset needed - WikiText? Something else because data leakage? TODO: Explore
# smashed_model_awq = AutoModelForCausalLM.from_pretrained(
#     temp_dir, quantization_config=awq_config, trust_remote_code=True
# )

Unused kwargs: ['skip_modules']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


AttributeError: 'BitsAndBytesConfig' object has no attribute 'skip_modules'

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Define model ID
model_id = "meta-llama/Meta-Llama-3-8B"

# Load tokenizer and model (assuming CUDA is available)
device = f"cuda:{0}"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# Input text
input_text = "Once upon a time"

# Convert input text to tensor and move to device
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# Generate text using beam search (modify parameters as needed)
generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

# Decode generated IDs back to text
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Print generated text
print("Generated text:", generated_text)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 8. Loading WikiText

In [ ]:
import os
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset


# TODO: This dataset merge all independent sentences and process them as a single batch sample.
#  This is not ideal since sometimes sentences might switch from a topic to another.
#  However, it was done similarly on Wanda and SparseGPT code.
class TextDataset(Dataset):
    def __init__(self, dataset, tokenizer, sequence_length=2048):
        self.tokenizer = tokenizer
        self.dataset=dataset
        self.texts = dataset["text"]
        tokenized_dataset = self.tokenizer(" ".join(dataset["text"]), return_tensors="pt")
        self.data = tokenized_dataset.input_ids[0, :-1]
        self.labels = tokenized_dataset.input_ids[0]
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.data) // self.sequence_length

    def __getitem__(self, index):
        start_index = index * self.sequence_length
        end_index = (index + 1) * self.sequence_length
        return self.data[start_index:end_index], self.labels[start_index + 1 : end_index + 1]


# TODO: This dataset look at each sentence individually as a batch sample.
#  This is not ideal since sometimes sentences might not switch from a topic to another.
# class TextDataset(Dataset):
#     def __init__(self, dataset, tokenizer, sequence_length=2048):
#         self.texts = dataset["text"]
#         self.tokenizer = tokenizer
#         tokenized_dataset = self.tokenizer(
#             self.texts, return_tensors="pt", truncation=True, padding=True, max_length=sequence_length
#         )
#         self.data = tokenized_dataset.input_ids
#         self.sequence_length = sequence_length
#
#     def __len__(self):
#         return len(self.data)
#
#     def __getitem__(self, index):
#         return self.data[index, :-1], self.data[index, 1:]


class WikiTextDataModule(LightningDataModule):
    def __init__(self, directory_dataset=os.getcwd(), batch_size=64, sequence_length=2048, tokenizer_name=None, seed=1):
        super().__init__()
        self.directory_dataset = directory_dataset
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, legacy=False)
        self.sequence_length = sequence_length
        self.prepare_data()

    def prepare_data(self):
        # Load train, val, and test datasets
        self.train_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        self.val_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
        self.test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

    # Use for calibration data
    def train_dataloader(self, batch_size=None, sequence_length=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        dataset = TextDataset(self.train_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length)
        train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        return train_dataloader

    # At this moment we are not using it
    def val_dataloader(self, batch_size=None, sequence_length=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        dataset = TextDataset(self.val_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length)
        val_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        return val_dataloader

    # Use for evaluating perplexity
    def test_dataloader(self, batch_size=None, sequence_length=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        dataset = TextDataset(self.test_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length)
        test_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        return test_dataloader

## 9. Text Streamer

In [14]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)